In [1]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

Found existing installation: keras 3.10.0
Uninstalling keras-3.10.0:
  Successfully uninstalled keras-3.10.0
Found existing installation: matplotlib 3.10.0
Uninstalling matplotlib-3.10.0:
  Successfully uninstalled matplotlib-3.10.0
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.simplefilter('ignore')

In [3]:
import os
import sys
import subprocess

In [4]:
def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony'
    ], check=True)

In [5]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if 'wheels' in f.lower() or f.endswith('.tar.gz'):
            print(os.path.join(root, f))

/kaggle/input/notebooks/andreasbis/aimo-3-utils/wheels.tar.gz


In [6]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if 'config' in f.lower() and f.endswith('.json'):
            print(os.path.join(root, f))

/kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1/config.json
/kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1/tokenizer_config.json
/kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1/generation_config.json


In [7]:
import os
for d in os.listdir('/kaggle/input'):
    print(d)
    path = f'/kaggle/input/{d}'
    if os.path.isdir(path):
        for f in os.listdir(path):
            print(f'  {f}')

models
  danielhanchen
competitions
  ai-mathematical-olympiad-progress-prize-3
notebooks
  andreasbis


In [8]:
set_env(
    input_archive='/kaggle/input/notebooks/andreasbis/aimo-3-utils/wheels.tar.gz',
    temp_dir='/kaggle/tmp/setup'
)

Looking in links: /kaggle/tmp/setup/wheels
Processing /kaggle/tmp/setup/wheels/unsloth-2025.12.9-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/trl-0.24.0-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/vllm-0.11.2-cp38-abi3-manylinux1_x86_64.whl
Processing /kaggle/tmp/setup/wheels/openai_harmony-0.0.8-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/tmp/setup/wheels/unsloth_zoo-2025.12.7-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/tyro-1.0.3-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/xformers-0.0.33.post1-cp39-abi3-manylinux_2_28_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/bitsandbytes-0.49.0-py3-none-manylinux_2_24_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/datasets-4.3.0-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/hf_transfer-0.1.9-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyldavis 3.4.1 requires scikit-learn>=1.0.0, which is not installed.
tpot 1.1.0 requires matplotlib>=3.6.2, which is not installed.
tpot 1.1.0 requires scikit-learn>=1.6, which is not installed.
ydata-profiling 4.18.1 requires matplotlib<=3.10,>=3.5, which is not installed.
librosa 0.11.0 requires scikit-learn>=1.1.0, which is not installed.
cuml-cu12 26.2.0 requires scikit-learn>=1.5, which is not installed.
shap 0.50.0 requires scikit-learn, which is not installed.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
bigframes 2.35.0 requires matplotlib>=3.7.1, which is not installed.
giddy 2.3.8 requires matplotlib, which is not installed.
sentence-transformers 5.2.3 requires scikit-learn, which is not installed.
pynndescent 0.6.0 requires scikit-learn>=0.18, which is 

In [9]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

cl100k_base.tiktoken
o200k_base.tiktoken


CompletedProcess(args=['ls', '/kaggle/tmp/setup/tiktoken_encodings'], returncode=0)

In [10]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [11]:
import gc
import re
import ast
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server


In [12]:
class CFG:

    system_prompt = (
        'You are a world-class International Mathematical Olympiad (IMO) competitor. '
        'Knowledge cutoff: 2024-06\n'
        'Current date: 2025-04-15\n'
        '\n'
        '# Rules\n'
        '- The final answer must be a non-negative integer between 0 and 99999.\n'
        '- You must place the final integer answer inside \\boxed{}.\n'
        '- Use Python tool calls to compute, verify, and check your work.\n'
        '- After finding your answer, write a short verification (substitution, brute-force, or formula check).\n'
        '\n'
        '# Approach\n'
        '1. Analyze the problem carefully before coding.\n'
        '2. Break complex problems into sub-problems.\n'
        '3. Use `math`, `numpy`, `sympy` as needed.\n'
        '4. Verify your answer with an independent check before boxing it.'
    )
    
    tool_prompt = (
        'Use this tool to execute Python code. '
        'The environment is a stateful Jupyter notebook. '
        'You must use print() to output results.'
    )
    
    preference_prompt = (
        'You have access to `math`, `numpy` and `sympy` to solve the problem. '
        'After finding your answer, verify it by writing a short Python check '
        '(substitution, brute-force small cases, or formula evaluation). '
        'For complex problems, consider breaking them into smaller sub-problems.'
    )

    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1'
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 270

    notebook_limit = 17400
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 6
    sandbox_timeout = 3

    stream_interval = 200
    context_tokens = 65536
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 256
    early_stop = 4  # Modified: now gated by constraint check before locking
    attempts = 8
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.96
    temperature = 1.0
    min_p = 0.02


In [13]:
set_seed(CFG.seed)

In [14]:
class AIMO3Template:

    def __init__(self):

        pass

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:

        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:

        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)

        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, user_message]

In [15]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import mpmath\n'
            'import itertools\n'
            'import collections\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):
        
        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import mpmath\n'
            'import itertools\n'
            'import collections\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):

        self.close()

In [16]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        
        self._owns_session = sandbox is None
        
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:

        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction, 
            tools=[]
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:

        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')

        if channel:
            message = message.with_channel(channel)

        return message

    def process_sync_plus(self, message: Message) -> list[Message]:

        self._ensure_session()
        raw_script = message.content[0].text
        final_script = self._ensure_last_print(raw_script)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        return [self._make_response(output, channel=message.channel)]

In [17]:
import ast

# ──────────────────────────────────────────────────────────────────────
# CHECK_RANK: used by pick_representative to rank certificate quality
# ──────────────────────────────────────────────────────────────────────
CHECK_RANK = {
    'bruteforce': 5, 'backward_verify': 5, 'substitution': 4,
    'formula_eval': 3, 'mod_check': 2, 'bound_check': 1, 'none': 0
}


# ──────────────────────────────────────────────────────────────────────
# normalize_output: semantic comparison of code execution outputs
# ──────────────────────────────────────────────────────────────────────
def normalize_output(output_text):
    """Normalize a code output for semantic comparison.
    
    Handles: list/set reordering, float→int coercion, whitespace.
    """
    text = output_text.strip()
    if not text:
        return text
    # For multi-line output, normalize only the last non-empty line
    lines = [l for l in text.split('\n') if l.strip()]
    if lines:
        text = lines[-1].strip()
    try:
        val = ast.literal_eval(text)
        if isinstance(val, complex):
            return str(val)
        if isinstance(val, (list, tuple)):
            try:
                return str(sorted(val))
            except TypeError:
                # unhashable or uncomparable elements
                return str(val)
        if isinstance(val, set):
            try:
                return str(sorted(val))
            except TypeError:
                return str(val)
        if isinstance(val, dict):
            try:
                return str(dict(sorted(val.items())))
            except TypeError:
                return str(val)
        if isinstance(val, float) and val == int(val) and not (val != val):  # not NaN
            return str(int(val))
        return str(val)
    except (ValueError, SyntaxError, TypeError, RecursionError):
        return text.strip()


# ──────────────────────────────────────────────────────────────────────
# extract_certificate: build structured cert from branch result
# ──────────────────────────────────────────────────────────────────────
def extract_certificate(result):
    """Extract verification certificate from a completed branch."""
    code_outputs = result.get('code_outputs', [])
    answer = result['Answer']
    
    total_errors = sum(1 for co in code_outputs if co['is_error'])
    total_calls = len(code_outputs)
    
    # Find the last successful tool output that produced a number
    last_tool_output = None
    for co in reversed(code_outputs):
        if not co['is_error']:
            try:
                nums = re.findall(r'\b(\d+)\b', co['output'])
                if nums:
                    last_tool_output = int(nums[-1])
                    break
            except (ValueError, IndexError):
                pass
    
    # Check if answer matches last tool output
    answer_matches_tool = (answer is not None and 
                          last_tool_output is not None and 
                          answer == last_tool_output)
    
    # Detect self-check code and classify its type
    check_code = None
    check_type = 'none'
    self_check_passed = None
    
    # Look for the step where the answer first appears in output
    answer_found_step = None
    if answer is not None:
        for co in code_outputs:
            if not co['is_error'] and co['output']:
                # Use word boundary to avoid partial matches (e.g. answer=1 matching "21")
                if re.search(rf'\b{re.escape(str(answer))}\b', co['output']):
                    answer_found_step = co['step']
                    break
    
    if answer_found_step is not None:
        for co in code_outputs:
            if co['step'] > answer_found_step and not co['is_error']:
                code = co['code']
                output = co['output']
                
                # Only consider if the code references the answer value
                # (helps filter continuation vs verification blocks)
                if str(answer) not in code and 'verify' not in code.lower() and 'check' not in code.lower():
                    continue
                
                # Classify check type
                code_lower = code.lower()
                if 'brute' in code_lower or ('range(' in code and 'for' in code) or 'itertools' in code:
                    check_type = 'bruteforce'
                elif any(w in code_lower for w in ['substitute', 'subs(', '.subs']):
                    check_type = 'substitution'
                elif 'solve' in code_lower and str(answer) in code:
                    check_type = 'backward_verify'
                elif 'eval' in code_lower or 'formula' in code_lower:
                    check_type = 'formula_eval'
                elif '%' in code or 'mod' in code_lower:
                    check_type = 'mod_check'
                
                check_code = code
                
                # Did the check pass?
                out_lower = output.lower().strip()
                if any(w in out_lower for w in ['true', 'verified', 'correct', 'pass', 'confirmed']):
                    self_check_passed = True
                elif any(w in out_lower for w in ['false', 'failed', 'incorrect', 'wrong']):
                    self_check_passed = False
                # else: None (ambiguous)
                
                break  # Take the first post-answer check
    
    return {
        'answer': answer,
        'branch_id': result['Attempt'],
        'self_check_passed': self_check_passed,
        'last_tool_output': last_tool_output,
        'total_errors': total_errors,
        'total_calls': total_calls,
        'error_rate': total_errors / max(total_calls, 1),
        'answer_matches_tool': answer_matches_tool,
        'check_code': check_code,
        'check_type': check_type,
        'best_entropy': result['Entropy'],
    }


# ──────────────────────────────────────────────────────────────────────
# pick_representative: best certificate for a set of same-answer branches
# ──────────────────────────────────────────────────────────────────────
def pick_representative(certificates):
    """Pick the best certificate for a group of branches with the same answer."""
    def sort_key(cert):
        return (
            cert['self_check_passed'] is True,       # prefer passed checks
            cert['answer_matches_tool'],              # prefer tool-consistent
            -cert['total_errors'],                    # fewer errors (neg for ascending)
            CHECK_RANK.get(cert['check_type'], 0),   # stronger check type
        )
    return max(certificates, key=sort_key)


# ──────────────────────────────────────────────────────────────────────
# tir_consensus: cross-branch intermediate computation comparison
# AUTHORITY: DIAGNOSTIC ONLY — does NOT kill or eliminate answers.
# Minority intermediate computations do NOT mean wrong answers.
# Used for: divergence detection, shared fact extraction, Try 2 hints.
# ──────────────────────────────────────────────────────────────────────
def tir_consensus(all_results):
    """
    DIAGNOSTIC ONLY — does not kill answers.
    
    Compare intermediate computations across branches.
    Returns dict with verified_intermediates, divergence_point, branch_verdicts.
    
    NOTE: Minority branch computations may still be correct. This signal
    is for divergence detection and hint generation, never elimination.
    """
    # 1. Collect all code+output pairs from branches with answers
    all_computations = []
    
    for result in all_results:
        if result['Answer'] is None:
            continue
        for co in result.get('code_outputs', []):
            if co['is_error']:
                continue
            code = co['code']
            output = co['output']
            
            if not output or not output.strip():
                continue
            
            # Extract ALL print() expressions (not just the first)
            # Only match simple print(expr) — skip formatted strings
            print_matches = re.finditer(r'print\(([^)]+)\)', code)
            for pm in print_matches:
                expr = pm.group(1).strip()
                # Skip formatted strings and complex expressions
                if expr.startswith(('f"', "f'", 'f"""', "f'''", '"', "'")):
                    continue
                # Skip very long expressions (likely not standalone computations)
                if len(expr) > 100:
                    continue
                    
                norm_out = normalize_output(output)
                all_computations.append({
                    'branch_id': result['Attempt'],
                    'expression': expr,
                    'normalized_output': norm_out,
                    'raw_output': output,
                    'code': code,
                })
    
    # 2. Group by expression (exact match on cleaned expression text)
    expr_groups = defaultdict(list)
    for comp in all_computations:
        key = comp['expression'].strip()
        expr_groups[key].append(comp)
    
    # 3. Find agreements and disagreements
    verified_intermediates = []
    divergence_point = None
    branch_wrong = set()
    
    for expr, comps in expr_groups.items():
        if len(comps) < 2:
            continue  # Need 2+ to compare
        
        output_counts = Counter(c['normalized_output'] for c in comps)
        most_common_output, most_common_count = output_counts.most_common(1)[0]
        
        if most_common_count == len(comps):
            # Full agreement — verified intermediate
            verified_intermediates.append({
                'name': expr,
                'value': most_common_output,
                'agreement_count': most_common_count,
            })
        else:
            # Disagreement — potential divergence point
            if divergence_point is None:
                divergence_point = f"compute {expr}"
            
            # Mark minority branches as wrong
            for c in comps:
                if c['normalized_output'] != most_common_output:
                    branch_wrong.add(c['branch_id'])
    
    # 4. Build per-branch verdicts
    branch_verdicts = {}
    for result in all_results:
        bid = result['Attempt']
        if bid in branch_wrong:
            branch_verdicts[bid] = 'wrong'
        else:
            branch_verdicts[bid] = 'correct'
    
    return {
        'verified_intermediates': sorted(
            verified_intermediates, 
            key=lambda x: -x['agreement_count']
        ),
        'divergence_point': divergence_point,
        'branch_verdicts': branch_verdicts,
    }


# ──────────────────────────────────────────────────────────────────────
# Gate 1: Branch Cleanliness Labels
# AUTHORITY: SOFT SIGNAL ONLY — labels branches, does NOT kill answers.
# Used as a medium-weight validator in decision rules and ranking.
# ──────────────────────────────────────────────────────────────────────
def gate_1_cleanliness(certificates):
    """
    SOFT SIGNAL — does not kill answers.
    
    Label each certificate as CLEAN or DIRTY.
    
    CLEAN if ALL true:
      - answer_matches_tool (dual extraction consistent)
      - total_errors == 0
      - self_check_passed is not False (True or None both OK)
    """
    for cert in certificates:
        is_clean = (
            cert['answer_matches_tool'] and
            cert['total_errors'] == 0 and
            cert['self_check_passed'] is not False  # None is OK
        )
        cert['is_clean'] = is_clean
    return certificates


# ──────────────────────────────────────────────────────────────────────
# Gate 2 helpers: hardcoded check detection + answer substitution
# ──────────────────────────────────────────────────────────────────────
def is_hardcoded_check(code, answer):
    """
    Detect if check code just compares against a hardcoded answer literal.
    E.g., `assert result == 437` with no real computation.
    Returns True if hardcoded (NOT a genuine constraint check).
    """
    if answer is None:
        return True  # Can't substitute into nothing
    answer_str = str(answer)
    # Escape for regex (answer could be a number with no special chars, but be safe)
    answer_esc = re.escape(answer_str)
    
    patterns = [
        rf'==\s*{answer_esc}\b',
        rf'!=\s*{answer_esc}\b',
        rf'\b{answer_esc}\s*==',
        rf'assert.*{answer_esc}',
    ]
    
    for pattern in patterns:
        if re.search(pattern, code):
            # If code uses sympy/numpy/math functions, it's doing real computation — genuine
            if any(lib in code for lib in ['sympy.', 'numpy.', 'np.', 'math.']):
                return False
            # Short code with answer comparison and no library = hardcoded
            lines = [l.strip() for l in code.split('\n') 
                     if l.strip() and not l.strip().startswith('#')]
            if len(lines) <= 3:
                return True
            # Longer code with comparison — ambiguous, treat as genuine
            return False
    
    return False


def substitute_answer_in_check(check_code, original_answer, new_answer):
    """
    Replace the original answer value in check code with a new candidate.
    Targets variable assignments like `answer = 437`, `result = 437`, etc.
    """
    code = check_code
    orig_str = str(original_answer)
    new_str = str(new_answer)
    orig_esc = re.escape(orig_str)
    
    # Replace assignments: answer = 437, result = 437, ans = 437, final_answer = 437
    code = re.sub(
        rf'\b(answer|result|ans|final_answer|n|x|val|value)\s*=\s*{orig_esc}\b',
        rf'\1 = {new_str}',
        code
    )
    
    # Also replace if the answer appears as a direct argument to check functions
    # e.g., check(437) → check(new_answer)
    # But only if it's a standalone number, not part of a larger expression
    code = re.sub(
        rf'(?<=[\(,\s]){orig_esc}(?=[\),\s])',
        new_str,
        code
    )
    
    return code


# ──────────────────────────────────────────────────────────────────────
# Independence helper: code skeleton extraction
# ──────────────────────────────────────────────────────────────────────
def code_skeleton(code):
    """
    Extract a structural skeleton from check code for independence dedup.
    Strips variable names, numeric literals, whitespace, and operator spacing
    so that near-duplicate checks hash to the same skeleton.
    
    Used by Gate 2 to ensure Gate 3 only counts truly independent failures.
    """
    if not code:
        return ''
    # Remove comments
    s = re.sub(r'#.*', '', code)
    # Normalize operator spacing: collapse spaces around all operators
    # so `answer = 42` and `answer=42` and `answer =42` all match.
    for op in ['==', '!=', '<=', '>=', '+=', '-=', '*=', '/=', '**', '//', 
               '=', '+', '-', '*', '/', '%', '<', '>', '(', ')', ',', ':', '[', ']']:
        escaped = re.escape(op)
        s = re.sub(rf'\s*{escaped}\s*', op, s)
    # Normalize remaining whitespace
    s = re.sub(r'\s+', ' ', s).strip()
    # Replace all numeric literals with placeholder
    s = re.sub(r'\b\d+\.?\d*\b', 'NUM', s)
    # Replace all identifier-like names with placeholder
    # Keep Python keywords and builtins intact
    keywords = {
        'False', 'None', 'True', 'and', 'as', 'assert', 'async', 'await',
        'break', 'class', 'continue', 'def', 'del', 'elif', 'else', 'except',
        'finally', 'for', 'from', 'global', 'if', 'import', 'in', 'is',
        'lambda', 'nonlocal', 'not', 'or', 'pass', 'raise', 'return', 'try',
        'while', 'with', 'yield', 'print', 'range', 'len', 'int', 'float',
        'str', 'list', 'dict', 'set', 'sum', 'abs', 'min', 'max', 'any',
        'all', 'enumerate', 'zip', 'map', 'filter', 'sorted', 'isinstance',
        'type', 'hasattr', 'getattr',
    }
    # Also keep library names
    libraries = {
        'sympy', 'numpy', 'np', 'math', 'itertools', 'functools',
        'collections', 'fractions', 'decimal', 're',
    }
    keep = keywords | libraries
    
    def replace_ident(m):
        word = m.group(0)
        return word if word in keep else 'VAR'
    
    s = re.sub(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', replace_ident, s)
    return s


# ──────────────────────────────────────────────────────────────────────
# Gate 2: Cross-Pollination + Backward Falsification
# AUTHORITY: SOFT SIGNAL ONLY — produces cross_passes/cross_fails counts.
# Does NOT kill answers. Feeds into Gate 3 which is the only kill gate.
# ──────────────────────────────────────────────────────────────────────
def gate_2_cross_pollination(certificates, all_results, sandbox_pool):
    """
    SOFT SIGNAL — does not kill answers.
    
    Run each branch's check code against every candidate answer.
    Record cross_passes, cross_fails, net_cross per answer.
    
    Independence dedup: checks with the same code skeleton are grouped.
    Only one check per unique skeleton contributes failures, preventing
    near-duplicate checks from overcounting in Gate 3.
    
    CRITICAL: Filter out hardcoded answer comparisons (not genuine checks).
    """
    candidate_answers = set(c['answer'] for c in certificates if c['answer'] is not None)
    
    if not candidate_answers:
        return {'cross_passes': {}, 'cross_fails': {}, 'net_cross': {}}
    
    cross_passes = defaultdict(int)
    cross_fails = defaultdict(int)
    # Track which checks passed on at least one answer (for validated check requirement)
    check_passed_any = {}  # check_id → bool
    check_failures = defaultdict(lambda: defaultdict(int))  # check_id → {answer: fail_count}
    
    # Independence dedup: only one check per unique code skeleton.
    # Prevents near-duplicate checks from overcounting failures in Gate 3.
    seen_skeletons = set()
    
    for cert_idx, cert in enumerate(certificates):
        check_code = cert.get('check_code')
        if check_code is None:
            continue
        
        if is_hardcoded_check(check_code, cert['answer']):
            continue
        
        # Independence check: skip if we've already seen this skeleton
        skeleton = code_skeleton(check_code)
        if skeleton in seen_skeletons:
            continue
        seen_skeletons.add(skeleton)
        
        check_id = f"check_{cert_idx}"
        check_passed_any[check_id] = False
        
        for candidate in candidate_answers:
            # FIX: Run the check against ALL candidates, including the branch's
            # own answer. Don't assume self-checks pass — actually verify.
            # For the self-answer, run the ORIGINAL code (no substitution needed).
            if candidate == cert['answer']:
                run_code = check_code  # original, unmodified
            else:
                run_code = substitute_answer_in_check(check_code, cert['answer'], candidate)
            
            sandbox = None
            try:
                sandbox = sandbox_pool.get(timeout=3)
            except Exception:
                continue
            
            try:
                result_text = sandbox.execute(run_code, timeout=5)
                out_lower = result_text.lower().strip()
                
                if any(w in out_lower for w in ['true', 'verified', 'correct', 'pass']):
                    if candidate != cert['answer']:
                        cross_passes[candidate] += 1
                    check_passed_any[check_id] = True
                elif any(w in out_lower for w in ['false', 'failed', 'incorrect', 'wrong', 'error',
                                                   'traceback', '[error]']):
                    if candidate != cert['answer']:
                        check_failures[check_id][candidate] += 1
                    # FIX: If the check fails even on its own answer, it's
                    # unreliable — don't mark it as validated.
                # else: ambiguous, don't count
            except Exception:
                pass
            finally:
                if sandbox is not None:
                    try:
                        sandbox.reset()
                    except Exception:
                        pass
                    sandbox_pool.put(sandbox)
    
    # Only count failures from VALIDATED checks (passed on at least one answer)
    for check_id, failures in check_failures.items():
        if check_passed_any.get(check_id, False):
            for answer, count in failures.items():
                cross_fails[answer] += count
    
    # Compute net cross-pollination
    net_cross = {}
    for answer in candidate_answers:
        net_cross[answer] = cross_passes.get(answer, 0) - cross_fails.get(answer, 0)
    
    return {
        'cross_passes': dict(cross_passes),
        'cross_fails': dict(cross_fails),
        'net_cross': net_cross,
    }


# ──────────────────────────────────────────────────────────────────────
# Gate 3: Constraint Elimination
# AUTHORITY: HARD KILL — the ONLY gate that can eliminate answers.
# Requires: deterministic, validated, independent failures.
# All other gates are soft signals that feed into this or into ranking.
# ──────────────────────────────────────────────────────────────────────
def gate_3_constraint_elimination(candidate_answers, cross_results, certificates):
    """
    HARD KILL — the only gate with elimination authority.
    
    Kill answers that fail 2+ independent validated constraint checks.
    Majority answers need 3+ failures.
    
    Independence is enforced upstream in Gate 2 via code_skeleton dedup.
    Validation is enforced upstream in Gate 2 via self-answer re-execution.
    """
    killed = set()
    
    if not candidate_answers:
        return killed
    
    branch_counts = Counter(c['answer'] for c in certificates if c['answer'] is not None)
    total_branches = sum(branch_counts.values())
    
    for answer in candidate_answers:
        if answer is None:
            continue
        
        fails = cross_results['cross_fails'].get(answer, 0)
        
        is_majority = branch_counts.get(answer, 0) > total_branches / 2
        threshold = 3 if is_majority else 2
        
        if fails >= threshold:
            killed.add(answer)
    
    # Range check: always kill out-of-range
    for answer in candidate_answers:
        if answer is not None and (answer < 0 or answer > 99999):
            killed.add(answer)
    
    return killed


# ──────────────────────────────────────────────────────────────────────
# Gate 4: Uniqueness Flag
# AUTHORITY: SOFT SIGNAL ONLY — flags, does NOT kill answers.
# Used as a medium-weight validator in decision rules and ranking.
# ──────────────────────────────────────────────────────────────────────
def gate_4_uniqueness(candidate_answers, check_codes, sandbox_pool):
    """
    SOFT SIGNAL — does not kill answers.
    
    Test neighboring values (V-1, V+1, V-2, V+2) against constraint checks.
    If only V passes and neighbors fail, UNIQUENESS_CONFIRMED.
    """
    uniqueness = {}
    
    if not check_codes:
        for answer in candidate_answers:
            uniqueness[answer] = 'NOT_APPLICABLE'
        return uniqueness
    
    for answer in candidate_answers:
        if answer is None:
            uniqueness[answer] = 'NOT_APPLICABLE'
            continue
        
        neighbors = [answer - 2, answer - 1, answer + 1, answer + 2]
        neighbors = [n for n in neighbors if 0 <= n <= 99999]
        
        neighbor_passes = 0
        neighbor_checks = 0
        
        for check_code in check_codes[:3]:  # Limit to 3 checks for performance
            for neighbor in neighbors:
                modified = substitute_answer_in_check(check_code, answer, neighbor)
                
                sandbox = None
                try:
                    sandbox = sandbox_pool.get(timeout=3)
                except Exception:
                    continue
                
                try:
                    result_text = sandbox.execute(modified, timeout=5)
                    neighbor_checks += 1
                    if any(w in result_text.lower() for w in ['true', 'verified', 'correct']):
                        neighbor_passes += 1
                except Exception:
                    pass
                finally:
                    if sandbox is not None:
                        try:
                            sandbox.reset()
                        except Exception:
                            pass
                        sandbox_pool.put(sandbox)
        
        if neighbor_checks == 0:
            uniqueness[answer] = 'NOT_APPLICABLE'
        elif neighbor_passes == 0:
            uniqueness[answer] = 'CONFIRMED'
        else:
            uniqueness[answer] = 'UNCERTAIN'
    
    return uniqueness


# ──────────────────────────────────────────────────────────────────────
# Decision Rules 1-5 (first match wins, no scoring)
# ──────────────────────────────────────────────────────────────────────
def apply_decision_rules(surviving_answers, gate_results, probe_result, branch_counts):
    """
    First match wins. No scoring, no weights.
    Returns (winning_answer, rule_number) or (None, None).
    
    Rule order per plan: 1, 2, 4, 5, 3.
    """
    if len(surviving_answers) == 1:
        return surviving_answers[0], 1
    
    if len(surviving_answers) == 0:
        return None, None
    
    # RULE 2: One answer has CLEAN branches (≥1 clean, ≥3 total)
    #         AND all others have ZERO clean branches
    clean_answers = [
        a for a in surviving_answers
        if gate_results.get(a, {}).get('clean_count', 0) >= 1 
        and branch_counts.get(a, 0) >= 3
    ]
    dirty_answers = [
        a for a in surviving_answers
        if gate_results.get(a, {}).get('clean_count', 0) == 0
    ]
    if len(clean_answers) == 1 and len(dirty_answers) == len(surviving_answers) - 1:
        return clean_answers[0], 2
    
    # RULE 4: One answer matches brute-force probe
    #         Fires if probe-match has ≥2 branches (hard floor).
    #         The half-rival condition is no longer an independent path
    #         because ceil(2/2) = 1, allowing 1-vs-2 flips.
    if probe_result is not None and isinstance(probe_result, list) and len(probe_result) > 0:
        matching = [a for a in surviving_answers if a in probe_result]
        if len(matching) == 1:
            probe_match = matching[0]
            probe_branches = branch_counts.get(probe_match, 0)
            # Hard floor: must have at least 2 branches.
            # This prevents 1-branch flukes from overriding anything.
            if probe_branches >= 2:
                return probe_match, 4
    
    # RULE 5: One answer has net positive cross-pollination, all others net negative
    positive_cross = [
        a for a in surviving_answers
        if gate_results.get(a, {}).get('net_cross', 0) > 0
    ]
    negative_cross = [
        a for a in surviving_answers
        if gate_results.get(a, {}).get('net_cross', 0) < 0
    ]
    if len(positive_cross) == 1 and len(negative_cross) == len(surviving_answers) - 1:
        return positive_cross[0], 5
    
    # RULE 3: One answer UNIQUENESS_CONFIRMED, others UNCERTAIN/NOT_APPLICABLE
    confirmed = [
        a for a in surviving_answers
        if gate_results.get(a, {}).get('uniqueness') == 'CONFIRMED'
    ]
    uncertain = [
        a for a in surviving_answers
        if gate_results.get(a, {}).get('uniqueness') in ('UNCERTAIN', 'NOT_APPLICABLE')
    ]
    if len(confirmed) == 1 and len(uncertain) == len(surviving_answers) - 1:
        return confirmed[0], 3
    
    return None, None


# ──────────────────────────────────────────────────────────────────────
# pick_best_candidate: rank surviving answers when no rule fires
# ──────────────────────────────────────────────────────────────────────
def pick_best_candidate(surviving_answers, representatives, gate_results, probe_result):
    """Pick best candidate from surviving answers when no rule fires."""
    if not surviving_answers:
        return None
    
    candidates = []
    for answer in surviving_answers:
        cert = representatives.get(answer)
        if cert is None:
            continue
        
        gates_passed = 0
        gr = gate_results.get(answer, {})
        
        if gr.get('clean_count', 0) > 0:
            gates_passed += 1
        # CHANGED: require net_cross >= 2 to count as a gate passed.
        # A single positive cross result is often noise; only a consistent
        # signal across multiple checks should count.
        if gr.get('net_cross', 0) >= 2:
            gates_passed += 1
        if gr.get('uniqueness') == 'CONFIRMED':
            gates_passed += 1
        
        # CHANGED: probe match is worth 2 points, not 1.
        # Probe is a direct computational verification — stronger signal than
        # a noisy positive cross result. This prevents a weak cross signal from
        # tying probe support, which lets entropy/branch_count pick the wrong majority.
        probe_match = False
        if probe_result is not None and isinstance(probe_result, list) and answer in probe_result:
            gates_passed += 2
            probe_match = True
        
        candidates.append({
            'answer': answer,
            'gates_passed': gates_passed,
            'probe_match': probe_match,
            'clean_count': gr.get('clean_count', 0),
            'entropy': cert.get('best_entropy', float('inf')),
            'branch_count': gr.get('total_branches', 0),
        })
    
    if not candidates:
        return None
    
    # Sort: gates_passed DESC, probe_match DESC (tiebreak before entropy),
    #        clean_count DESC, entropy ASC, branch_count DESC
    candidates.sort(key=lambda c: (
        -c['gates_passed'],
        -int(c['probe_match']),
        -c['clean_count'],
        c['entropy'],
        -c['branch_count'],
    ))
    
    return candidates[0]


# ──────────────────────────────────────────────────────────────────────
# final_fallback: Choose between Try 1 and Try 2 best candidates
# ──────────────────────────────────────────────────────────────────────
def final_fallback(try1_best, try2_best, killed_answers):
    """
    Pick between Try 1 and Try 2 best candidates.
    Try 1 wins ties (less biased — no wrong-answer hints).
    Returns the answer (int) or None.
    """
    if try1_best is None and try2_best is None:
        return None
    
    if try1_best is None:
        ans = try2_best['answer']
        return ans if ans not in killed_answers else None
    
    if try2_best is None:
        ans = try1_best['answer']
        return ans if ans not in killed_answers else None
    
    t1 = try1_best
    t2 = try2_best
    
    # Skip killed answers
    t1_alive = t1['answer'] not in killed_answers
    t2_alive = t2['answer'] not in killed_answers
    
    if t1_alive and not t2_alive:
        return t1['answer']
    if t2_alive and not t1_alive:
        return t2['answer']
    if not t1_alive and not t2_alive:
        return None
    
    # Both alive — compare
    # More gates passed wins
    if t1['gates_passed'] > t2['gates_passed']:
        return t1['answer']
    if t2['gates_passed'] > t1['gates_passed']:
        return t2['answer']
    
    # More clean branches wins
    if t1['clean_count'] > t2['clean_count']:
        return t1['answer']
    if t2['clean_count'] > t1['clean_count']:
        return t2['answer']
    
    # Lower entropy wins (outside tolerance band)
    if abs(t1['entropy'] - t2['entropy']) > 0.1:
        if t1['entropy'] < t2['entropy']:
            return t1['answer']
        else:
            return t2['answer']
    
    # Tie: Try 1 wins (less biased)
    return t1['answer']


# ──────────────────────────────────────────────────────────────────────
# build_try2_input: construct the modified prompt for Try 2
# ──────────────────────────────────────────────────────────────────────
def build_try2_input(problem_text, preference_prompt, gate3_killed, 
                     verified_intermediates, divergence_point, constraint_facts=None):
    """Build Try 2 problem input with progressive hints."""
    input_text = problem_text
    
    # Level 1: Wrong answers (bare fact, no rationale)
    if gate3_killed:
        killed_list = ', '.join(str(a) for a in gate3_killed)
        input_text += f'\n\nVerified incorrect: {killed_list}.'
    
    # Level 2: Verified intermediates (max 5, most-agreed first)
    if verified_intermediates:
        sorted_facts = sorted(verified_intermediates,
                              key=lambda x: -x['agreement_count'])
        facts = '; '.join(f'{f["name"]} = {f["value"]}' for f in sorted_facts[:5])
        input_text += f'\n\nEstablished facts: {facts}.'
    
    # Level 3: Constraint facts from Gate 3
    if constraint_facts:
        input_text += f'\n\n{constraint_facts}'
    
    # Level 4: Divergence point
    if divergence_point:
        input_text += (
            f'\n\nPrevious attempts agreed on the above facts '
            f'but diverged when trying to {divergence_point}.'
        )
    
    input_text += f' {preference_prompt}'
    return input_text


# ──────────────────────────────────────────────────────────────────────
# classify_problem: rule-based problem classification (zero cost)
# ──────────────────────────────────────────────────────────────────────
def classify_problem(problem_text):
    """
    Rule-based classification. Zero cost (regex/keyword matching).
    Returns set of categories.
    """
    categories = set()
    text_lower = problem_text.lower()
    
    if any(w in text_lower for w in ['prime', 'divisor', 'gcd', 'lcm', 'modulo',
                                       'congruent', 'remainder', 'factor']):
        categories.add('number_theory')
    
    if any(w in text_lower for w in ['recurrence', 'sequence', 'fibonacci',
                                       'a_n', 'a_{n', 'recursive']):
        categories.add('recurrence')
    
    if any(w in text_lower for w in ['polynomial', 'root', 'equation', 'vieta',
                                       'quadratic', 'cubic']):
        categories.add('algebra')
    
    if any(w in text_lower for w in ['triangle', 'circle', 'area', 'angle',
                                       'perpendicular', 'bisector', 'circumscribe']):
        categories.add('geometry')
    
    if any(w in text_lower for w in ['how many', 'count', 'number of ways',
                                       'permutation', 'combination', 'choose']):
        categories.add('combinatorics')
    
    if any(w in text_lower for w in ['find', 'determine', 'compute', 'calculate',
                                       'what is', 'evaluate']):
        categories.add('find_value')
    
    return categories if categories else {'general'}


# ──────────────────────────────────────────────────────────────────────
# quick_constraint_check: fast sanity check for early stop gating
# ──────────────────────────────────────────────────────────────────────
def quick_constraint_check(answer, problem_text):
    """
    Quick constraint check: extract simple falsifiable constraints from the
    problem text and test the answer against them.
    
    Must be FAST (<1 sec) since it runs during branch collection.
    Returns False if definitely wrong, None if can't tell, True if passes all.
    """
    if answer is None:
        return False
    if answer < 0 or answer > 99999:
        return False
    
    text_lower = problem_text.lower()
    
    # ── Parity constraints ──
    # "find the even number..." / "the answer is odd"
    if re.search(r'\b(?:find|determine|compute|what is)\b.*\beven\b.*\b(?:integer|number|value)\b', text_lower):
        if answer % 2 != 0:
            return False
    if re.search(r'\b(?:find|determine|compute|what is)\b.*\bodd\b.*\b(?:integer|number|value)\b', text_lower):
        if answer % 2 != 1:
            return False
    
    # ── Modular residue: "remainder when ... divided by N" ──
    mod_match = re.search(
        r'remainder\s+(?:when\s+)?.*?(?:divided|dividing)\s+by\s+(\d+)', text_lower
    )
    if mod_match:
        modulus = int(mod_match.group(1))
        if modulus > 0 and answer >= modulus:
            # The remainder must be < modulus
            return False
    
    # ── "Find the last three digits" → answer < 1000 ──
    if re.search(r'last\s+(?:three|3)\s+digits', text_lower):
        if answer >= 1000:
            return False
    if re.search(r'last\s+(?:two|2)\s+digits', text_lower):
        if answer >= 100:
            return False
    if re.search(r'last\s+(?:four|4)\s+digits', text_lower):
        if answer >= 10000:
            return False
    
    # ── Divisibility: "answer is divisible by N" / "N divides the answer" ──
    div_match = re.search(
        r'(?:divisible\s+by|multiple\s+of)\s+(\d+)', text_lower
    )
    if div_match:
        divisor = int(div_match.group(1))
        if divisor > 0 and answer % divisor != 0:
            return False
    
    # ── Explicit upper bound: "where N < 1000" / "less than 1000" ──
    bound_match = re.search(
        r'(?:less\s+than|smaller\s+than|<)\s*(\d+)', text_lower
    )
    if bound_match:
        bound = int(bound_match.group(1))
        if answer >= bound:
            return False
    
    # If we checked at least one constraint and none failed, return True.
    # If no constraints were extractable, return None (can't tell).
    has_constraints = any([
        mod_match,
        div_match,
        bound_match,
        re.search(r'last\s+(?:two|three|four|2|3|4)\s+digits', text_lower),
        re.search(r'\b(?:find|determine|compute|what is)\b.*\b(?:even|odd)\b.*\b(?:integer|number|value)\b', text_lower),
    ])
    
    return True if has_constraints else None


In [18]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):
    
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
    
        self._preload_model_weights()
        
        self.server_process = self._start_server()
    
        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )
    
        self._wait_for_server()
        self._initialize_kernels()
    
        self.notebook_start_time = time.time()
        self.problems_remaining = 50
    
    def _preload_model_weights(self) -> None:
    
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0
    
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
    
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)
    
        def _read_file(path: str) -> None:
    
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))
    
        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')
    
    def _start_server(self) -> subprocess.Popen:
    
        cmd = [
            sys.executable, 
            '-m', 
            'vllm.entrypoints.openai.api_server', 
            '--seed', 
            str(self.cfg.seed), 
            '--model', 
            self.cfg.model_path, 
            '--served-model-name', 
            self.cfg.served_model_name, 
            '--tensor-parallel-size', 
            '1', 
            '--max-num-seqs', 
            str(self.cfg.batch_size), 
            '--gpu-memory-utilization', 
            str(self.cfg.gpu_memory_utilization), 
            '--host', 
            '0.0.0.0', 
            '--port', 
            str(self.port), 
            '--dtype', 
            self.cfg.dtype, 
            '--kv-cache-dtype', 
            self.cfg.kv_cache_dtype, 
            '--max-model-len', 
            str(self.cfg.context_tokens), 
            '--stream-interval', 
            str(self.cfg.stream_interval), 
            '--async-scheduling', 
            '--disable-log-stats', 
            '--enable-prefix-caching'
        ]
    
        self.log_file = open('vllm_server.log', 'w')
    
        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )
    
    def _wait_for_server(self):
    
        print('Waiting for vLLM server...')
        start_time = time.time()
    
        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
    
            if return_code is not None:
                self.log_file.flush()
    
                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()
    
                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')
    
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
    
                return
    
            except Exception:
                time.sleep(1)
    
        raise RuntimeError('Server failed to start (timeout).\n')
    
    def _initialize_kernels(self) -> None:
    
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()
    
        self.sandbox_pool = queue.Queue()
    
        def _create_sandbox():
            
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]
    
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())
    
        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')
    
    def _scan_for_answer(self, text: str) -> int | None:
        
        pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
                
        pattern = r'final\s+answer\s+is\s*([0-9,]+)'
        matches = re.findall(pattern, text, re.IGNORECASE)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
    
        return None
    
    def _compute_mean_entropy(self, logprobs_buffer: list) -> float:
    
        if not logprobs_buffer:
            return float('inf')
    
        total_entropy = 0.0
        token_count = 0
    
        for top_logprobs_dict in logprobs_buffer:
            
            if not isinstance(top_logprobs_dict, dict):
                continue
            
            if not top_logprobs_dict:
                continue
            
            token_entropy = 0.0
            
            for token_str, log_prob in top_logprobs_dict.items():
                prob = math.exp(log_prob)
                
                if prob > 0:
                    token_entropy -= prob * math.log2(prob)
            
            total_entropy += token_entropy
            token_count += 1
    
        if token_count == 0:
            return float('inf')
    
        return total_entropy / token_count

    # ──────────────────────────────────────────────────────────────────
    # _process_attempt: MODIFIED — code capture, error correction,
    #   kill rules, temperature parameter, expanded return
    # ──────────────────────────────────────────────────────────────────
    def _process_attempt(
        self, 
        problem: str, 
        system_prompt: str, 
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float,
        temperature: float = None,  # NEW: per-branch temperature
    ) -> dict:

        # Use provided temperature or fall back to config default
        effective_temperature = temperature if temperature is not None else self.cfg.temperature
    
        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1, 
                'Answer': None, 
                'Python Calls': 0, 
                'Python Errors': 0, 
                'Response Length': 0, 
                'Entropy': float('inf'),
                'code_outputs': [],
                'logprobs_buffer': [],
                'consecutive_errors_at_end': 0,
                'killed': False,
                'kill_reason': None,
            }
    
        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        
        logprobs_buffer = []
        code_outputs = []                  # NEW: passive capture of all code executions
        consecutive_errors = 0             # NEW: track consecutive error streak
        last_code_token_count = 0          # NEW: for drift detection
        full_response_text_parts = []      # NEW: for loop detection across turns
        killed = False
        kill_reason = None
        post_answer_turns = 0              # FIX: allow verification turns after answer found
        POST_ANSWER_BUDGET = 2            # FIX: max extra turns for certificate capture
    
        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))
    
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
    
            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout, 
                tool_prompt=self.cfg.tool_prompt, 
                sandbox=sandbox
            )
    
            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt, 
                problem, 
                local_tool.tool_config
            )
    
            conversation = Conversation.from_messages(messages)
    
            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break
    
                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)
    
                if max_tokens < self.cfg.buffer_tokens:
                    break
    
                stream = self.client.completions.create(
                    model=self.cfg.served_model_name, 
                    temperature=effective_temperature,   # CHANGED: use effective_temperature
                    logprobs=self.cfg.top_logprobs, 
                    max_tokens=max_tokens, 
                    prompt=prompt_ids, 
                    seed=attempt_seed, 
                    stream=True, 
                    extra_body={
                        'min_p': self.cfg.min_p, 
                        'stop_token_ids': self.stop_token_ids, 
                        'return_token_ids': True
                    }
                )
    
                try:
                    token_buffer = []
                    text_chunks = []
    
                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break
    
                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text
    
                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)
                            
                            chunk_logprobs = chunk.choices[0].logprobs
                            
                            if chunk_logprobs is not None:
                                if chunk_logprobs.top_logprobs:
                                    logprobs_buffer.extend(chunk_logprobs.top_logprobs)
    
                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(search_text)
    
                            if answer is not None:
                                final_answer = answer
                                # FIX: do NOT break here — let the stream finish so
                                # the full token_buffer is captured for message parsing.
                                # Verification/check code may follow in subsequent turns.
    
                finally:
                    stream.close()
    
                # Accumulate text for loop detection
                if text_chunks:
                    full_response_text_parts.extend(text_chunks)
    
                # FIX: Instead of breaking immediately when answer is found,
                # allow up to POST_ANSWER_BUDGET more turns so verification
                # code blocks (certificates) can still execute and be captured.
                if final_answer is not None:
                    post_answer_turns += 1
                    if post_answer_turns > POST_ANSWER_BUDGET:
                        break
    
                if not token_buffer:
                    break
    
                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]
    
                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    if final_answer is None:
                        final_answer = self._scan_for_answer(answer_text)
                    break
    
                if last_message.recipient == 'python':
                    python_calls += 1
                    
                    # Capture raw code BEFORE process_sync_plus (which may modify it)
                    raw_script = last_message.content[0].text
                    
                    tool_responses = local_tool.process_sync_plus(last_message)
                    response_text = tool_responses[0].content[0].text
    
                    # DRY error detection — compute once, reuse
                    is_error = (
                        response_text.startswith('[ERROR]') or 
                        'Traceback' in response_text or 
                        'Error:' in response_text
                    )

                    # Capture code output (NEW)
                    code_outputs.append({
                        'code': raw_script,
                        'output': response_text,
                        'step': python_calls,
                        'is_error': is_error
                    })

                    # Structured error correction (NEW)
                    if is_error:
                        python_errors += 1
                        consecutive_errors += 1
                        
                        if consecutive_errors == 1:
                            # Give structured correction guidance
                            error_msg = (
                                f'{response_text}\n\n'
                                'Fix the code error above and re-run. '
                                'Do not change your mathematical approach.'
                            )
                            tool_responses = [local_tool._make_response(
                                error_msg, channel=last_message.channel
                            )]
                        
                        elif consecutive_errors >= 3:
                            # Kill branch: stuck in error loop
                            killed = True
                            kill_reason = 'consecutive_errors'
                            conversation.messages.extend(tool_responses)
                            break
                        
                        # consecutive_errors == 2: pass through raw response (baseline behavior)
                    else:
                        consecutive_errors = 0  # Reset on any successful execution
    
                    conversation.messages.extend(tool_responses)
                    
                    # Update last code position for drift detection
                    last_code_token_count = total_tokens

                # ── Branch kill: drift detection ──
                # If 2000+ tokens since last code block, and we've used code before
                if (total_tokens - last_code_token_count) > 2000 and python_calls > 0:
                    killed = True
                    kill_reason = 'drift'
                    break
                
                # ── Branch kill: reasoning loop detection ──
                if len(full_response_text_parts) > 20:
                    recent_text = ''.join(full_response_text_parts[-10:])
                    if len(recent_text) >= 500:
                        # Check if last 500 chars appear earlier (limit search window)
                        earlier_text = ''.join(full_response_text_parts[:-10])
                        search_window = earlier_text[-5000:] if len(earlier_text) > 5000 else earlier_text
                        if recent_text[-500:] in search_window:
                            killed = True
                            kill_reason = 'loop'
                            break
    
        except Exception as exc:
            python_errors += 1
    
        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)
    
        mean_entropy = self._compute_mean_entropy(logprobs_buffer)
    
        return {
            'Attempt': attempt_index + 1, 
            'Response Length': total_tokens, 
            'Python Calls': python_calls, 
            'Python Errors': python_errors, 
            'Entropy': mean_entropy, 
            'Answer': final_answer,
            'code_outputs': code_outputs,                      # NEW
            'logprobs_buffer': logprobs_buffer,                # NEW
            'consecutive_errors_at_end': consecutive_errors,   # NEW
            'killed': killed,                                  # NEW
            'kill_reason': kill_reason,                        # NEW
        }

    # ──────────────────────────────────────────────────────────────────
    # _baseline_select_answer_with_gate3: baseline voting minus killed
    # ──────────────────────────────────────────────────────────────────
    def _baseline_select_answer_with_gate3(self, detailed_results, killed_answers):
        """Baseline entropy-weighted voting, but skip Gate-3-killed answers."""
        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)

        for result in detailed_results:
            answer = result['Answer']
            entropy = result['Entropy']

            if answer is not None and answer not in killed_answers:
                weight = 1.0 / max(entropy, 1e-9)
                answer_weights[answer] += weight
                answer_votes[answer] += 1

        if not answer_weights:
            # Everything killed — last resort: use original baseline without gate3
            return self._select_answer(detailed_results)

        scored = sorted(answer_weights.items(), key=lambda x: -x[1])
        return scored[0][0]

    # ──────────────────────────────────────────────────────────────────
    # _select_answer: KEPT as ultimate fallback (renamed internally)
    # ──────────────────────────────────────────────────────────────────
    def _select_answer(self, detailed_results: list) -> int:

        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)

        for result in detailed_results:
            answer = result['Answer']
            entropy = result['Entropy']
            
            if answer is not None:
                weight = 1.0 / max(entropy, 1e-9)
                
                answer_weights[answer] += weight
                answer_votes[answer] += 1

        scored_answers = []

        for answer, total_weight in answer_weights.items():
            scored_answers.append({
                'answer': answer, 
                'votes': answer_votes[answer], 
                'score': total_weight
            })

        scored_answers.sort(key=lambda x: x['score'], reverse=True)

        vote_data = []

        for item in scored_answers:
            vote_data.append((
                item['answer'], 
                item['votes'], 
                item['score']
            ))

        vote_dataframe = pd.DataFrame(
            vote_data, 
            columns=['Answer', 'Votes', 'Score']
        )

        vote_dataframe = vote_dataframe.round({'Score': 3})
        display(vote_dataframe)
        
        if not scored_answers:
            print('\nFinal Answer: 0\n')
            return 0

        final_answer = scored_answers[0]['answer']    
        print(f'\nFinal Answer: {final_answer}\n')

        return final_answer

    # ──────────────────────────────────────────────────────────────────
    # _run_branches: reusable parallel branch execution
    # ──────────────────────────────────────────────────────────────────
    def _run_branches(self, user_input, system_prompt, n_branches,
                      temperature, deadline, stop_event=None, seed_offset=0,
                      probe_result=None):
        """Run N branches in parallel. Returns list of result dicts."""
        if stop_event is None:
            stop_event = threading.Event()
        
        detailed_results = []
        valid_answers = []
        
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
        try:
            futures = []
            for i in range(n_branches):
                future = executor.submit(
                    self._process_attempt,
                    user_input, system_prompt, i + seed_offset,
                    stop_event, deadline, temperature,
                )
                futures.append(future)
            
            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
                    
                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])
                    
                    # Early stop check with constraint gating
                    counts = Counter(valid_answers).most_common()
                    if counts:
                        consensus = counts[0][0]
                        consensus_count = counts[0][1]
                        
                        # Probe-aware early stop policy:
                        # 1. If probe agrees or is absent → default threshold (4)
                        # 2. If probe disagrees AND a probe-backed minority answer
                        #    has already appeared → DISABLE early stop entirely.
                        #    The whole point of the probe is to rescue a minority;
                        #    locking in the majority defeats that.
                        # 3. If probe disagrees but no probe-backed answer has
                        #    appeared yet → raise threshold by 1 to give it a chance.
                        
                        probe_active = (probe_result and isinstance(probe_result, list)
                                        and len(probe_result) > 0)
                        probe_agrees = probe_active and consensus in probe_result
                        
                        if probe_active and not probe_agrees:
                            # Check if a probe-backed minority is already present
                            probe_minority_present = any(
                                a in probe_result for a in valid_answers
                                if a != consensus
                            )
                            if probe_minority_present:
                                # Fully disable: let all branches run
                                pass  # skip early stop check entirely
                            else:
                                # Probe disagrees but its answer hasn't appeared yet.
                                # Raise threshold by 1 to give it a chance.
                                threshold = self.cfg.early_stop + 1
                                if consensus_count >= threshold:
                                    constraint_ok = quick_constraint_check(consensus, user_input)
                                    if constraint_ok is not False:
                                        stop_event.set()
                                        for f in futures:
                                            f.cancel()
                                        break
                        else:
                            # Probe agrees or absent → standard threshold
                            if consensus_count >= self.cfg.early_stop:
                                constraint_ok = quick_constraint_check(consensus, user_input)
                                if constraint_ok is not False:
                                    stop_event.set()
                                    for f in futures:
                                        f.cancel()
                                    break
                            # If falsified: DON'T stop. Let remaining branches run.
                    
                except Exception as exc:
                    print(f'Branch failed: {exc}')
        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)
        
        return detailed_results

    # ──────────────────────────────────────────────────────────────────
    # _run_branches_mixed_temp: run branches with per-branch temps
    # ──────────────────────────────────────────────────────────────────
    def _run_branches_mixed_temp(self, user_input, system_prompt, temps,
                                  deadline, seed_offset=0):
        """Run branches with mixed temperatures. Returns list of result dicts."""
        stop_event = threading.Event()
        detailed_results = []
        valid_answers = []
        
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
        try:
            futures = []
            for i, temp in enumerate(temps):
                future = executor.submit(
                    self._process_attempt,
                    user_input, system_prompt, i + seed_offset,
                    stop_event, deadline, temp,
                )
                futures.append(future)
            
            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
                    
                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])
                except Exception as exc:
                    print(f'Branch failed: {exc}')
        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)
        
        return detailed_results

    # ──────────────────────────────────────────────────────────────────
    # _extract_all_certificates: build certs from all branch results
    # ──────────────────────────────────────────────────────────────────
    def _extract_all_certificates(self, all_results):
        """Extract certificates from all branch results."""
        certificates = []
        for result in all_results:
            try:
                cert = extract_certificate(result)
                certificates.append(cert)
            except Exception as exc:
                print(f'Certificate extraction failed for attempt {result.get("Attempt", "?")}: {exc}')
        return certificates

    # ──────────────────────────────────────────────────────────────────
    # _run_all_gates: orchestrate Gates 1-4
    # ──────────────────────────────────────────────────────────────────
    def _run_all_gates(self, certificates, all_results, tir_result):
        """Run all 4 gates and return combined results."""
        
        # Gate 1: cleanliness labels (modifies certificates in-place)
        gate_1_cleanliness(certificates)
        
        # Aggregate per answer
        answer_certs = defaultdict(list)
        for cert in certificates:
            if cert['answer'] is not None:
                answer_certs[cert['answer']].append(cert)
        
        # Pick representatives
        representatives = {}
        for answer, certs in answer_certs.items():
            representatives[answer] = pick_representative(certs)
        
        # Gate 2: cross-pollination
        try:
            cross_results = gate_2_cross_pollination(certificates, all_results, self.sandbox_pool)
        except Exception as exc:
            print(f'Gate 2 error: {exc}')
            cross_results = {'cross_passes': {}, 'cross_fails': {}, 'net_cross': {}}
        
        # Gate 3: constraint elimination
        candidate_answers = list(answer_certs.keys())
        killed = gate_3_constraint_elimination(candidate_answers, cross_results, certificates)
        
        # Gate 4: uniqueness (only for top 3 candidates, performance guard)
        check_codes = [c['check_code'] for c in certificates if c.get('check_code')]
        top_candidates = sorted(candidate_answers, 
                                key=lambda a: len(answer_certs.get(a, [])), 
                                reverse=True)[:3]
        try:
            uniqueness = gate_4_uniqueness(top_candidates, check_codes[:3], self.sandbox_pool)
        except Exception as exc:
            print(f'Gate 4 error: {exc}')
            uniqueness = {a: 'NOT_APPLICABLE' for a in candidate_answers}
        
        # Combine gate results
        gate_results = {'killed': killed}
        for answer in candidate_answers:
            gate_results[answer] = {
                'clean_count': sum(1 for c in answer_certs[answer] if c.get('is_clean')),
                'total_branches': len(answer_certs[answer]),
                'net_cross': cross_results['net_cross'].get(answer, 0),
                'cross_passes': cross_results['cross_passes'].get(answer, 0),
                'cross_fails': cross_results['cross_fails'].get(answer, 0),
                'uniqueness': uniqueness.get(answer, 'NOT_APPLICABLE'),
            }
        
        return gate_results, representatives

    # ──────────────────────────────────────────────────────────────────
    # brute_force_probe: quick computational probe
    # ──────────────────────────────────────────────────────────────────
    def brute_force_probe(self, problem_text):
        """
        Quick computational probe. Temp 0.3, 20 sec timeout.
        Returns list of 0-3 candidate values.
        """
        probe_prompt = (
            "Write Python code to brute-force or symbolically compute the "
            "answer. Try small cases, exhaustive search, SymPy solving. "
            "Print only the final integer answer."
        )
        
        full_prompt = f"{problem_text}\n\n{probe_prompt}"
        
        sandbox = None
        try:
            sandbox = self.sandbox_pool.get(timeout=3)
        except Exception:
            return []
        
        try:
            local_tool = AIMO3Tool(
                local_jupyter_timeout=15,  # strict timeout
                tool_prompt=self.cfg.tool_prompt,
                sandbox=sandbox,
            )
            
            messages = self.template.apply_chat_template(
                self.cfg.system_prompt, full_prompt, local_tool.tool_config
            )
            conversation = Conversation.from_messages(messages)
            
            probe_start = time.time()
            candidates = []
            
            for turn in range(3):
                if time.time() - probe_start > 20:
                    break
                
                prompt_ids = self.encoding.render_conversation_for_completion(
                    conversation, Role.ASSISTANT
                )
                max_tokens = min(4096, self.cfg.context_tokens - len(prompt_ids))
                
                if max_tokens < self.cfg.buffer_tokens:
                    break
                
                # Use streaming (safer compatibility with vLLM)
                stream = self.client.completions.create(
                    model=self.cfg.served_model_name,
                    temperature=0.3,
                    max_tokens=max_tokens,
                    prompt=prompt_ids,
                    seed=self.cfg.seed,
                    stream=True,
                    extra_body={
                        'min_p': self.cfg.min_p,
                        'stop_token_ids': self.stop_token_ids,
                        'return_token_ids': True,
                    }
                )
                
                token_buffer = []
                text_chunks = []
                
                try:
                    for chunk in stream:
                        if time.time() - probe_start > 20:
                            break
                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text
                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            text_chunks.append(new_text)
                        
                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(search_text)
                            if answer is not None:
                                candidates.append(answer)
                                break
                finally:
                    stream.close()
                
                if candidates:
                    break
                
                if not token_buffer:
                    break
                
                # Parse response for tool calls
                new_messages = self.encoding.parse_messages_from_completion_tokens(
                    token_buffer, Role.ASSISTANT
                )
                conversation.messages.extend(new_messages)
                last = new_messages[-1]
                
                if last.recipient == 'python':
                    responses = local_tool.process_sync_plus(last)
                    conversation.messages.extend(responses)
                    
                    output = responses[0].content[0].text
                    nums = re.findall(r'\b(\d{1,5})\b', output)
                    for n in nums[-3:]:
                        val = int(n)
                        if 0 <= val <= 99999:
                            candidates.append(val)
                elif last.channel == 'final':
                    answer_text = last.content[0].text
                    answer = self._scan_for_answer(answer_text)
                    if answer is not None:
                        candidates.append(answer)
                    break
                else:
                    break
            
            return list(set(candidates))[:3]
        
        except Exception as exc:
            print(f'Probe error: {exc}')
            return []
        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)

    # ──────────────────────────────────────────────────────────────────
    # _run_try2: execute Try 2 branches with hints
    # ──────────────────────────────────────────────────────────────────
    def _run_try2(self, problem_text, tir_result, gate3_killed, remaining, deadline):
        """Run Try 2 with progressive hints and mixed temperatures."""
        # Branch count proportional to remaining time:
        # ~1 branch per 40 seconds, clamped to [4, 16].
        # This scales smoothly instead of using fixed step thresholds.
        n_branches = max(4, min(16, int(remaining / 40)))
        print(f'Try 2: {n_branches} branches for {remaining:.0f}s remaining')
        
        # Temperature assignment: 75% at 1.0, 25% at 0.6
        n_low_temp = n_branches // 4
        n_high_temp = n_branches - n_low_temp
        temps = [1.0] * n_high_temp + [0.6] * n_low_temp
        
        # Build Try 2 input with hints
        try2_input = build_try2_input(
            problem_text, self.cfg.preference_prompt,
            gate3_killed,
            tir_result.get('verified_intermediates', []),
            tir_result.get('divergence_point'),
            None,  # constraint_facts
        )
        
        return self._run_branches_mixed_temp(
            try2_input, self.cfg.system_prompt,
            temps=temps, deadline=deadline,
            seed_offset=100,  # Different seeds from Try 1
        )

    # ──────────────────────────────────────────────────────────────────
    # oss_critique_tiebreak: rare LLM-as-judge tiebreak
    # AUTHORITY: WEAK SIGNAL — last-resort tiebreaker only.
    # Cannot override deterministic evidence (probe, Gate 3).
    # Used only when all surviving answers are in the same confidence tier.
    # ──────────────────────────────────────────────────────────────────
    def oss_critique_tiebreak(self, candidate_a, candidate_b, cert_a, cert_b, budget):
        """
        WEAK SIGNAL — last-resort tiebreaker only.
        
        Rare tiebreak: two candidates survived all gates.
        Ask the model to prove each one correct.
        Cannot override deterministic evidence (probe, Gate 3 kills).
        """
        if budget < 90:
            return None
        
        prompt = (
            f"Two candidates remain after verification.\n\n"
            f"Candidate A: answer={candidate_a}, check_result={cert_a.get('self_check_passed')}\n"
            f"Candidate B: answer={candidate_b}, check_result={cert_b.get('self_check_passed')}\n\n"
            f"For each candidate, attempt to construct a short proof that it "
            f"is correct. Write a Python test to verify your proof. Run it.\n\n"
            f"If one proof succeeds and the other fails, that determines the answer.\n"
            f"Output \\boxed{{A}} or \\boxed{{B}} or \\boxed{{UNCERTAIN}}"
        )
        
        try:
            result = self._process_attempt(
                prompt, self.cfg.system_prompt,
                attempt_index=99,
                stop_event=threading.Event(),
                deadline=time.time() + 30,
                temperature=0.2,
            )
            
            # Parse response
            answer_val = result.get('Answer')
            if answer_val is not None:
                # Model output a number — match to candidates
                if answer_val == candidate_a:
                    return candidate_a
                elif answer_val == candidate_b:
                    return candidate_b
            
            # Check text output in code_outputs for A/B/UNCERTAIN
            for co in result.get('code_outputs', []):
                out_upper = co['output'].upper()
                if 'UNCERTAIN' in out_upper:
                    return None
                if 'CANDIDATE A' in out_upper or f'{candidate_a}' in co['output']:
                    return candidate_a
                if 'CANDIDATE B' in out_upper or f'{candidate_b}' in co['output']:
                    return candidate_b
        except Exception:
            pass
        
        return None  # Safe default

    # ──────────────────────────────────────────────────────────────────
    # solve_problem: RESTRUCTURED — multi-stage pipeline
    # ──────────────────────────────────────────────────────────────────
    def solve_problem(self, problem: str) -> int:
    
        print(f'\nProblem: {problem}\n')
        problem_start = time.time()
        
        user_input = f'{problem} {self.cfg.preference_prompt}'
    
        # ── Budget computation (KEPT from baseline) ──
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout
    
        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)
    
        deadline = time.time() + budget
    
        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')

        # Track all results across tries for baseline fallback
        all_detailed_results = []
        killed_answers = set()
        try1_best = None
        try2_best = None
        rule_used = None
        try2_triggered = False
        probe_result = []

        try:
            # ── STEP 0: Brute-force probe (20 sec max) ──
            try:
                probe_result = self.brute_force_probe(problem)
                if probe_result:
                    print(f'Probe candidates: {probe_result}')
            except Exception as exc:
                print(f'Probe failed: {exc}')
                probe_result = []

            # ── STEP 1: Try 1 — 8 branches at temp 1.0 ──
            try1_results = self._run_branches(
                user_input, self.cfg.system_prompt,
                n_branches=self.cfg.attempts, temperature=self.cfg.temperature,
                deadline=deadline, probe_result=probe_result
            )
            all_detailed_results.extend(try1_results)

            # Display results table (kept from baseline)
            if try1_results:
                display_data = [{
                    'Attempt': r['Attempt'],
                    'Response Length': r['Response Length'],
                    'Python Calls': r['Python Calls'],
                    'Python Errors': r['Python Errors'],
                    'Entropy': round(r['Entropy'], 3),
                    'Answer': r['Answer'],
                    'Killed': r.get('killed', False),
                    'Kill Reason': r.get('kill_reason', ''),
                } for r in try1_results]
                results_dataframe = pd.DataFrame(display_data)
                results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')
                display(results_dataframe)

            valid_answers = [r['Answer'] for r in try1_results if r['Answer'] is not None]
            
            if not valid_answers:
                print('\nNo valid answers from Try 1. Result: 0\n')
                return 0

            # ── STEP 2: Post-processing — certificates + TIR + gates ──
            certificates = self._extract_all_certificates(try1_results)
            
            tir_result = tir_consensus(try1_results)
            if tir_result['divergence_point']:
                print(f"TIR divergence: {tir_result['divergence_point']}")
            
            gate_results, representatives = self._run_all_gates(
                certificates, try1_results, tir_result
            )
            killed_answers = gate_results.get('killed', set())
            if killed_answers:
                print(f'Gate 3 killed: {killed_answers}')

            # ── STEP 3: Decision rules ──
            branch_counts = Counter(c['answer'] for c in certificates if c['answer'] is not None)
            surviving = [a for a in branch_counts.keys() if a not in killed_answers]
            
            winner, rule_used = apply_decision_rules(
                surviving, gate_results, probe_result, branch_counts
            )
            
            if winner is not None:
                print(f'Rule {rule_used} fired: answer={winner}, '
                      f'time={time.time() - problem_start:.1f}s')
                return winner

            # ── STEP 4: Save Try 1 best ──
            try1_best = pick_best_candidate(
                surviving, representatives, gate_results, probe_result
            )
            if try1_best:
                print(f'Try 1 best: {try1_best["answer"]} (gates={try1_best["gates_passed"]})')

            # ── STEP 5: Try 2 (if budget allows and we have uncertainty) ──
            remaining = deadline - time.time()
            if remaining > 300 and try1_best is not None:
                try2_triggered = True
                print(f'\nTry 2: {remaining:.0f}s remaining')
                
                try:
                    try2_results = self._run_try2(
                        problem, tir_result, killed_answers, remaining, deadline
                    )
                    all_detailed_results.extend(try2_results)
                    
                    # Process Try 2 results
                    try2_certs = self._extract_all_certificates(try2_results)
                    gate_1_cleanliness(try2_certs)
                    
                    # Merge certificates for combined analysis
                    all_certs = certificates + try2_certs
                    
                    # Re-aggregate per answer
                    all_answer_certs = defaultdict(list)
                    for cert in all_certs:
                        if cert['answer'] is not None:
                            all_answer_certs[cert['answer']].append(cert)
                    
                    # Re-pick representatives
                    all_representatives = {}
                    for answer, certs in all_answer_certs.items():
                        all_representatives[answer] = pick_representative(certs)
                    
                    # Combined branch counts
                    combined_counts = Counter(c['answer'] for c in all_certs if c['answer'] is not None)
                    combined_surviving = [a for a in combined_counts.keys() if a not in killed_answers]
                    
                    # Build combined gate results
                    combined_gate_results = {'killed': killed_answers}
                    for answer in combined_counts.keys():
                        certs_for_ans = all_answer_certs[answer]
                        combined_gate_results[answer] = {
                            'clean_count': sum(1 for c in certs_for_ans if c.get('is_clean')),
                            'total_branches': len(certs_for_ans),
                            'net_cross': gate_results.get(answer, {}).get('net_cross', 0),
                            'cross_passes': gate_results.get(answer, {}).get('cross_passes', 0),
                            'cross_fails': gate_results.get(answer, {}).get('cross_fails', 0),
                            'uniqueness': gate_results.get(answer, {}).get('uniqueness', 'NOT_APPLICABLE'),
                        }
                    
                    # Re-run decision rules on combined data
                    winner2, rule2 = apply_decision_rules(
                        combined_surviving, combined_gate_results, probe_result, combined_counts
                    )
                    
                    if winner2 is not None:
                        rule_used = rule2
                        print(f'Combined Rule {rule2} fired: answer={winner2}, '
                              f'time={time.time() - problem_start:.1f}s')
                        return winner2
                    
                    try2_best = pick_best_candidate(
                        combined_surviving, all_representatives, combined_gate_results, probe_result
                    )
                    if try2_best:
                        print(f'Try 2 best: {try2_best["answer"]} (gates={try2_best["gates_passed"]})')
                
                except Exception as exc:
                    print(f'Try 2 failed: {exc}')

            # ── STEP 6: Final fallback ──
            fallback_answer = final_fallback(try1_best, try2_best, killed_answers)
            
            if fallback_answer is not None:
                print(f'Fallback winner: {fallback_answer}, '
                      f'time={time.time() - problem_start:.1f}s')
                return fallback_answer
            
            # Ultimate fallback: baseline voting with gate3 safety
            print('Using baseline voting with gate3 safety...')
            baseline_answer = self._baseline_select_answer_with_gate3(
                all_detailed_results, killed_answers
            )
            
            print(f'Baseline fallback: {baseline_answer}, '
                  f'time={time.time() - problem_start:.1f}s')
            return baseline_answer

        except Exception as exc:
            print(f'solve_problem error: {exc}')
            # Even if everything fails, try baseline
            if all_detailed_results:
                return self._baseline_select_answer_with_gate3(all_detailed_results, killed_answers)
            return 0

        finally:
            self.problems_remaining = max(0, self.problems_remaining - 1)
            
            # Enhanced cleanup
            try:
                import requests
                requests.post(f'{self.base_url}/reset_prefix_cache')
            except Exception:
                pass
            
            elapsed = time.time() - problem_start
            print(f'Problem solved: rule={rule_used}, try2={try2_triggered}, '
                  f'killed={len(killed_answers)}, time={elapsed:.1f}s')
    
    def __del__(self):
    
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
    
        if hasattr(self, 'log_file'):
            self.log_file.close()
    
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
    
                except Exception:
                    pass


In [19]:
solver = AIMO3Solver(CFG)

Loading model weights from /kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1 into OS Page Cache...
Processed 26 files (65.28 GB) in 90.38 seconds.

Waiting for vLLM server...
Server is ready (took 135.93 seconds).

Initializing 16 persistent Jupyter kernels...
Kernels initialized in 3.06 seconds.



In [20]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    
    id_value = id_.item(0)
    question_text = question.item(0)
    
    gc.disable()
    
    final_answer = solver.solve_problem(question_text)
    
    gc.enable()
    gc.collect()
    
    return pl.DataFrame({'id': id_value, 'answer': final_answer})

In [21]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
    
else:
    inference_server.run_local_gateway(
        ('/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv',) 
    )


Problem: Solve $4+x=4$ for $x$.

Budget: 900.00 seconds | Deadline: 1775904161.70

Probe candidates: [0]


,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer,Killed,Kill Reason
0,1,222,0,0,0.376,0,False,None
1,4,230,0,0,0.416,0,False,None
2,5,226,1,0,0.508,0,False,None
3,6,236,0,0,0.440,0,False,None


Rule 1 fired: answer=0, time=19.2s
Problem solved: rule=1, try2=False, killed=0, time=19.2s

Problem: What is $1-1$?

Budget: 900.00 seconds | Deadline: 1775904181.05

Probe candidates: [0]


,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer,Killed,Kill Reason
0,8,178,0,0,0.479,0,False,None
1,4,190,0,0,0.473,0,False,None
2,1,208,0,0,0.347,0,False,None
3,6,213,0,0,0.423,0,False,None


Rule 1 fired: answer=0, time=3.2s
Problem solved: rule=1, try2=False, killed=0, time=3.2s

Problem: What is $0\times10$?

Budget: 900.00 seconds | Deadline: 1775904184.37

Probe candidates: [0]


,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer,Killed,Kill Reason
0,4,165,0,0,0.508,0,False,None
1,7,164,0,0,0.487,0,False,None
2,8,173,0,0,0.501,0,False,None
3,1,206,0,0,0.510,0,False,None


Rule 1 fired: answer=0, time=4.2s
Problem solved: rule=1, try2=False, killed=0, time=4.2s
